# Moore & Fallah (2001): analysis of one session

Computes the paper's measures from one run of `target-dimming-detection`:

1. **Each staircase, block by block** (p. 1274): the dimming magnitude (% Michelson contrast), the
   blink hits and held catch trials, and the block verdict (≥ 1 of 2 blinks detected and the catch
   trial held lowers the dimming one step).
2. **Luminance-change threshold** per staircase: the least-squares fit of `y = a·x / (b + x)`
   (p. 1274). `y` is read as the staircase's progress below its starting level, so the threshold is
   `start − a` (an ASSUMED reading, docs/methods.md).
3. **Threshold reduction with stimulation** (Fig. 3 legend, p. 1275): `(1 − stim / control) × 100`.
4. **Performance**: % correct on stimulation and control trials (the paper's first measure, p. 1275),
   hit, early-response and false-alarm rates, reaction times.
5. **Stimulation timing**: the realised stimulation-onset asynchrony, from the flip stamps of
   `STIM_TRIGGER` and `DIM_ON`.

The paper's comparison is across sessions (77 experiments at 34 sites, with the target inside or
outside the movement field). This notebook analyses one run; pool runs by concatenating their
results. A simulated run has no frontal eye field: its simulated observer ignores stimulation, so its
two thresholds differ only by chance.

Set `ANALYSIS_RUN` to a run folder, or leave it unset to use the newest run under `../data`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from moore_fallah2001.analysis import (
    block_table,
    fit_asymptote,
    load_run,
    performance,
    threshold_reduction_pct,
)

run = os.environ.get("ANALYSIS_RUN")
if run is None:
    runs = sorted(Path("../data").rglob("*_trials.csv"), key=lambda p: p.stat().st_mtime)
    if not runs:
        raise SystemExit("no run found: set ANALYSIS_RUN to a run folder")
    run = runs[-1].parent
run = Path(run)
trials, paradigm = load_run(run)
print(run.name, "|", len(trials), "attempts,", int(trials["completed"].sum()), "completed")
trials["outcome"].value_counts()

## 1. The staircases, block by block

In [ ]:
blocks = block_table(trials)
if paradigm is not None:
    check = paradigm.merge(blocks, on=["staircase", "block"], suffixes=("_scheduler", "_trials"))
    assert (check["block_success_scheduler"] == check["block_success_trials"]).all(), "scheduler and trials disagree"
blocks

## 2. Thresholds: least-squares fit of y = a·x / (b + x)

In [ ]:
start = float(blocks.sort_values("block").groupby("staircase")["dim_pct"].first().max())
fits = {}
for label, g in paradigm.sort_values("block").groupby("staircase"):
    levels = g["next_dim_pct"].to_numpy()
    fits[label] = fit_asymptote(levels, start) if len(levels) >= 2 else None
table = pd.DataFrame([
    {"staircase": k, "blocks": f.n_blocks, "a (progress, %)": f.a, "b (blocks)": f.b,
     "threshold (% Michelson)": f.threshold_pct if f.converged else np.nan,
     "levelled off": f.converged,
     "final level": paradigm.query("staircase == @k")["next_dim_pct"].iloc[-1]}
    for k, f in fits.items() if f is not None
])
if len(table) and not table["levelled off"].all():
    print("note: a staircase had not levelled off within its blocks; its threshold is not reported.")
if len(table) and table["blocks"].min() < 10:
    print(f"note: {table['blocks'].min()} blocks per staircase; the paper fitted 10-25.")
table

In [ ]:
colors = {"stim": "#8a1c1c", "control": "#3d3d3d"}
fig, ax = plt.subplots(figsize=(6.5, 3.6))
for label, g in paradigm.sort_values("block").groupby("staircase"):
    x = np.r_[0, g["block"].to_numpy()]
    y = np.r_[start, g["next_dim_pct"].to_numpy()]
    ax.step(x, y, where="post", color=colors.get(label, "C0"), label=f"{label} (staircase)")
    f = fits.get(label)
    if f is not None and f.converged:
        xx = np.linspace(0, x.max(), 200)
        ax.plot(xx, start - f.progress(xx), ls="--", color=colors.get(label, "C0"), lw=1,
                label=f"{label} fit, threshold {f.threshold_pct:.1f}%")
ax.set_xlabel("block")
ax.set_ylabel("dimming (% Michelson contrast)")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()

## 3. Threshold reduction with stimulation

In [ ]:
both = [fits.get("stim"), fits.get("control")]
if all(f is not None and f.converged for f in both):
    red = threshold_reduction_pct(fits["stim"].threshold_pct, fits["control"].threshold_pct)
    field = trials["target_field"].iloc[0]
    print(f"target {field} the movement field: threshold reduction = {red:.1f}% "
          "(positive = lower threshold with stimulation)")
elif fits.get("stim") is None:
    print("one staircase only (no stimulation in this run): no reduction to compute")
else:
    print("a staircase did not level off: no threshold, so no reduction")

## 4. Performance on stimulation and control trials

In [ ]:
performance(trials)

In [ ]:
hits = trials[trials["outcome"] == "HIT"]
fig, ax = plt.subplots(figsize=(5, 3))
for label, g in hits.groupby("staircase"):
    ax.hist(g["rt_ms"], bins=np.arange(0, 850, 50), alpha=0.6, label=label, color=colors.get(label, "C0"))
ax.set_xlabel("reaction time after the dimming (ms)")
ax.set_ylabel("hits")
ax.legend(frameon=False)
fig.tight_layout()

## 5. Stimulation timing

In [ ]:
soa = trials["realised_soa_ms"].dropna() if "realised_soa_ms" in trials else pd.Series(dtype=float)
planned = trials["planned_soa_ms"].dropna().unique() if "planned_soa_ms" in trials else []
print("planned SOA (whole frames, ms):", planned)
soa.describe()